In [ ]:
%load_ext manim

C:\Users\tmari\AppData\Roaming\Python\Python312\site-packages\pydub\utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)


The manim module is not an IPython extension.


In [2]:
%%manim -v WARNING -qm FeasibleRegionBounds

from manim import *

class FeasibleRegionBounds(Scene):
    def construct(self):
        # Coordinate system without labels
        ax = Axes(
            x_range=[0, 6, 1],
            y_range=[0, 6, 1],
            axis_config={"include_numbers": False},
            tips=False
        )

        # Define polygon (feasible region) via corner points
        vertices = [
            ax.coords_to_point(1, 1),
            ax.coords_to_point(1, 4),
            ax.coords_to_point(5, 4),
            ax.coords_to_point(5, 1),
        ]

        polygon = Polygon(*vertices, fill_color=BLUE, fill_opacity=0.4, stroke_color=BLUE)
        self.play(Create(ax), FadeIn(polygon))
        self.wait(0.5)

        # Mark lower bound (min point) and upper bound (max point)
        lower_dot = Dot(ax.coords_to_point(1, 1), color=RED)
        upper_dot = Dot(ax.coords_to_point(5, 4), color=GREEN)

        lower_label = MathTex(r"\text{Lower Bound}").next_to(lower_dot, DOWN)
        upper_label = MathTex(r"\text{Upper Bound}").next_to(upper_dot, UP)

        self.play(FadeIn(lower_dot), Write(lower_label))
        self.play(FadeIn(upper_dot), Write(upper_label))
        self.wait(2)


Manim Community v0.19.0

In [22]:
%%manim -v WARNING -qm FeasibleRegionWithAssumption

from manim import *

class FeasibleRegionWithAssumption(Scene):
    def construct(self):
        # 1. Axes
        ax = Axes(
            x_range=[0, 6, 1],
            y_range=[0, 6, 1],
            axis_config={"include_numbers": False},
            tips=False
        )
        self.play(Create(ax))

        # 2. Original feasible region (rectangle)
        original_vertices = [
            ax.coords_to_point(1, 1),
            ax.coords_to_point(1, 4),
            ax.coords_to_point(5, 4),
            ax.coords_to_point(5, 1),
        ]
        polygon = Polygon(*original_vertices, fill_color=BLUE, fill_opacity=0.4, stroke_color=BLUE)
        self.play(FadeIn(polygon))

        # 3. Original bounds
        lower_dot = Dot(ax.coords_to_point(1, 1), color=RED)
        upper_dot = Dot(ax.coords_to_point(5, 4), color=GREEN)
        lower_label = MathTex(r"\text{Lower Bound}").scale(0.6).next_to(lower_dot, DOWN)
        upper_label = MathTex(r"\text{Upper Bound}").scale(0.6).next_to(upper_dot, UP)
        self.play(FadeIn(lower_dot), Write(lower_label))
        self.play(FadeIn(upper_dot), Write(upper_label))
        self.wait(0.5)

        # 4. Diagonal line as an 'assumption'
        # Goes from (0,6) to (6,0) – just below the original top
        assumption_line = ax.plot_line_graph(
            x_values=[0, 8],
            y_values=[8, 0],
            line_color=YELLOW
        )
        self.play(Create(assumption_line))
        self.wait(1)

        # 5. Remove area above the line (simulate it disappearing)
        new_vertices = [
            ax.coords_to_point(1, 1), #bottom left
            ax.coords_to_point(1, 4), #top left
            ax.coords_to_point(4, 4), #new top right (1) touches assumption line
            ax.coords_to_point(5, 3),  #new top right (2) touches assumption line
            ax.coords_to_point(5, 1), #bottom right
        ]
        new_polygon = Polygon(*new_vertices, fill_color=BLUE, fill_opacity=0.4, stroke_color=BLUE)

        self.play(Transform(polygon, new_polygon), FadeOut(upper_dot), FadeOut(upper_label))
        self.wait(0.5)

        # 6. New upper bound
        new_upper = Dot(ax.coords_to_point(4, 4), color=GREEN)
        new_label = MathTex(r"\text{New Upper Bound}").scale(0.6).next_to(new_upper, UP+RIGHT)
        self.play(FadeIn(new_upper), Write(new_label))
        self.wait(2)


Manim Community v0.19.0

In [20]:
%%manim -v WARNING -qm BoundsWithObjectiveAndAssumption
from manim import *
import numpy as np

class BoundsWithObjectiveAndAssumption(Scene):
    def construct(self):
        # --- Axes (unlabeled) ---
        ax = Axes(
            x_range=[0, 6, 1],
            y_range=[0, 6, 1],
            axis_config={"include_numbers": False},
            tips=False
        )

        # --- Feasible region (rectangle) ---
        xL, xR = 1, 5
        yB, yT = 1, 4
        R_pts = [
            ax.coords_to_point(xL, yB),
            ax.coords_to_point(xL, yT),
            ax.coords_to_point(xR, yT),
            ax.coords_to_point(xR, yB),
        ]
        poly = Polygon(*R_pts, fill_color=BLUE, fill_opacity=0.4, stroke_color=BLUE)

        # 1) Draw axes and polygon
        self.play(Create(ax))
        self.play(FadeIn(poly))

        # --- Objective: y = m_obj*x + b  (NEGATIVE slope) ---
        m_obj = -0.6
        # Start roughly in the middle of the polygon
        b_start = (yT + yB)/2 - m_obj * ((xL + xR)/2)
        b = ValueTracker(b_start)

        def obj_line():
            return ax.plot(lambda x: m_obj*x + b.get_value(), x_range=[0, 6])

        line = always_redraw(obj_line)

        # (1) LABEL THAT MOVES WITH THE OBJECTIVE
        line_lbl = MathTex(r"\text{Objective (ATE/PNS)}").scale(0.6)
        anchor_x = xL + 0.5  # place label near the left side to avoid other text
        def update_label(mob):
            y = m_obj * anchor_x + b.get_value()
            mob.next_to(ax.coords_to_point(anchor_x, y), 0.5*UP+2*RIGHT, buff=0.2)
        line_lbl.add_updater(update_label)

        # 2) Objective line appears (label will track automatically)
        self.play(Create(line), FadeIn(line_lbl))

        # Helper
        def b_value(x, y): return y - m_obj*x
        rect_vertices_xy = [(xL, yB), (xL, yT), (xR, yT), (xR, yB)]

        # 3) Move to minimum + show Lower Bound
        min_xy = min(rect_vertices_xy, key=lambda p: b_value(*p))
        b_min = b_value(*min_xy)
        self.play(b.animate.set_value(b_min), run_time=1.2)
        lower_dot = Dot(ax.coords_to_point(*min_xy), color=RED)
        lower_lbl = MathTex(r"\text{Lower Bound}").scale(0.6).next_to(lower_dot, DOWN)
        self.play(FadeIn(lower_dot), Write(lower_lbl))

        # 4) Move to maximum + show Upper Bound (keep line there)
        max_xy = max(rect_vertices_xy, key=lambda p: b_value(*p))
        b_max = b_value(*max_xy)
        self.play(b.animate.set_value(b_max), run_time=1.2)
        upper_dot = Dot(ax.coords_to_point(*max_xy), color=GREEN)
        upper_lbl = MathTex(r"\text{Upper Bound}").scale(0.6).next_to(upper_dot, UP)
        self.play(FadeIn(upper_dot), Write(upper_lbl))

        # 5) Assumption (POSITIVE slope) appears and polytope is redrawn (clipped below the line)
        m_a = 0.2
        b_a = 2.0
        A = ax.coords_to_point(0, m_a*0 + b_a)
        B = ax.coords_to_point(6, m_a*6 + b_a)
        assumption = DashedLine(A, B, color=YELLOW, dash_length=0.2)

        # (2) LABEL NEAR THE Y-INTERCEPT OF ASSUMPTION
        a_lbl = MathTex(r"\text{Assumption}").scale(0.6)
        y_intercept = np.clip(b_a, yB, yT)  # keep inside the visible box
        a_lbl.next_to(ax.coords_to_point(0, y_intercept), RIGHT+0.5*DOWN, buff=0.2)

        self.play(Create(assumption), FadeIn(a_lbl))

        # Intersections of y = m_a x + b_a with rectangle
        y_left   = m_a * xL + b_a
        y_right  = m_a * xR + b_a
        x_top    = (yT - b_a) / m_a if m_a != 0 else float("inf")
        x_bottom = (yB - b_a) / m_a if m_a != 0 else float("inf")

        cross = []
        if yB <= y_left  <= yT:   cross.append((xL, y_left))
        if yB <= y_right <= yT:   cross.append((xR, y_right))
        if xL <= x_top   <= xR:   cross.append((x_top, yT))
        if xL <= x_bottom<= xR:   cross.append((x_bottom, yB))
        cross = list({(round(x,6), round(y,6)) for x,y in cross})
        cross.sort(key=lambda p: p[0])
        assert len(cross) == 2, f"Expected 2 intersections, got {cross}"
        (xi1, yi1), (xi2, yi2) = cross

        # Keep area BELOW the assumption line
        new_vertices_xy = [
            (xL, yB),
            (xi1, min(max(yi1, yB), yT)),
            (xi2, min(max(yi2, yB), yT)),
            (xR, yB),
        ]
        new_vertices = [ax.coords_to_point(x, y) for x, y in new_vertices_xy]
        poly_new = Polygon(*new_vertices, fill_color=BLUE, fill_opacity=0.4, stroke_color=BLUE)
        self.play(FadeOut(poly), FadeIn(poly_new))
        self.play(FadeOut(upper_dot), FadeOut(upper_lbl))

        # 6) Move objective to NEW max in the clipped polygon
        best_xy = max(new_vertices_xy, key=lambda p: b_value(*p))
        b_new_max = b_value(*best_xy)
        self.play(b.animate.set_value(b_new_max), run_time=1.2)

        # 7) Show New Upper Bound
        new_upper = Dot(ax.coords_to_point(*best_xy), color=GREEN)
        new_upper_lbl = MathTex(r"\text{New Upper Bound}").scale(0.6).next_to(new_upper, UP)
        self.play(FadeIn(new_upper), Write(new_upper_lbl))
        self.wait(2)


Manim Community v0.19.0